In [1]:
import pandas as pd
import re
import numpy as np
import google.generativeai as genai
import os
import csv
from dotenv import load_dotenv

load_dotenv()

c:\Users\anara\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
provas = pd.read_csv("C:/Users/anara/Desktop/Pesquisa - Tutoria/feedback-open-source/resultados-consolidados.csv", sep=None, engine="python")

In [3]:
provas_sample = provas.sample(10, random_state=123)

In [4]:
provas_sample['concept'].unique()

array(['reação', 'sistema nervoso', 'metabolismo', 'filtro', 'leucócitos',
       'epidídimo'], dtype=object)

In [5]:
for idx, row in provas_sample.iterrows():
    print(idx, row["concept"])

print(len(provas_sample))

49 reação
85 sistema nervoso
34 metabolismo
235 sistema nervoso
237 metabolismo
362 metabolismo
107 filtro
93 leucócitos
258 epidídimo
11 metabolismo
10


In [6]:
rubric_eval_prompt = """
Você é um auditor independente especialista em avaliação de feedback educacional
gerado por modelos de linguagem.

Sua tarefa é avaliar a qualidade de um feedback fornecido a um estudante,
considerando:

1. A descrição da atividade;
2. Os objetivos, critérios ou rubrica da atividade;
3. A resposta submetida pelo estudante;
4. O feedback gerado para essa resposta.

Você NÃO deve reescrever o feedback.
Você NÃO deve produzir um novo feedback.
Você NÃO deve sugerir melhorias fora dos campos solicitados.
Você deve apenas classificar o feedback nas 16 dimensões abaixo.

Analise cada dimensão de forma independente.

==================================================
DADOS PARA ANÁLISE
==================================================

DESCRIÇÃO DA ATIVIDADE:
{descricao_atividade}

OBJETIVOS OU RUBRICA DA ATIVIDADE:
{rubrica_atividade}

RESPOSTA DO ESTUDANTE:
{resposta_estudante}

FEEDBACK AVALIADO:
{feedback}

==================================================
PARTE 1 — CONTEÚDO DO FEEDBACK
==================================================

As dimensões de conteúdo devem ser avaliadas em uma escala de 0 a 2.

0 = dimensão ausente ou inadequada
1 = dimensão parcialmente presente
2 = dimensão claramente presente e bem aplicada

1. ALINHAMENTO COM OS OBJETIVOS

Avalia até que ponto os comentários do feedback estão relacionados aos
objetivos, critérios ou requisitos específicos da atividade.

Score 0:
O feedback não relaciona seus comentários aos objetivos ou critérios da atividade.

Score 1:
O feedback menciona os objetivos ou critérios de maneira parcial, vaga ou genérica.

Score 2:
A maior parte dos comentários está claramente relacionada aos objetivos,
critérios ou requisitos específicos da atividade.

2. ESPECIFICIDADE

Avalia se o feedback apresenta orientações detalhadas, concretas e acionáveis.

Score 0:
O feedback é genérico, vago ou não explica como o estudante pode melhorar.

Score 1:
O feedback apresenta alguma orientação útil, mas ainda é pouco detalhado
ou não fornece exemplos concretos.

Score 2:
O feedback oferece orientações claras, detalhadas e acionáveis, incluindo
exemplos ou indicações concretas de como melhorar.

3. TOM MOTIVACIONAL

Avalia se o tom é positivo, encorajador, respeitoso e apropriado ao contexto educacional.

Score 0:
O feedback utiliza tom negativo, desrespeitoso, desencorajador ou inadequado.

Score 1:
O feedback apresenta tom neutro ou parcialmente encorajador, mas de maneira inconsistente.

Score 2:
O feedback mantém um tom positivo, respeitoso, encorajador e construtivo.

4. IDENTIFICAÇÃO DE PONTOS FORTES

Avalia se o feedback identifica aspectos específicos que foram bem realizados
na resposta do estudante.

Score 0:
Não menciona nenhum ponto forte da resposta.

Score 1:
Menciona um ponto forte de maneira genérica, vaga ou pouco relacionada
à resposta do estudante.

Score 2:
Identifica claramente um ou mais pontos fortes específicos da resposta,
explicando por que estão adequados.

5. IDENTIFICAÇÃO DE PONTOS FRACOS

Avalia se o feedback identifica aspectos específicos que precisam ser corrigidos
ou aprimorados.

Score 0:
Não identifica problemas, limitações ou aspectos que precisam melhorar.

Score 1:
Menciona problemas de maneira vaga, genérica ou pouco explicada.

Score 2:
Identifica claramente um ou mais pontos fracos específicos e explica
o que precisa ser aprimorado.

6. LEGIBILIDADE

Avalia se o feedback é claro, compreensível, bem organizado e adequado
ao nível esperado do estudante.

Score 0:
O feedback é confuso, desorganizado, excessivamente complexo ou difícil de compreender.

Score 1:
O feedback é parcialmente compreensível, mas contém problemas de organização,
clareza, linguagem ou excesso de complexidade.

Score 2:
O feedback é claro, organizado, objetivo e facilmente compreensível pelo estudante.

==================================================
PARTE 2 — EFETIVIDADE DO FEEDBACK
==================================================

As dimensões de efetividade devem ser avaliadas de forma binária.

0 = o componente não está presente
1 = o componente está presente

Considere que uma dimensão somente deve receber 1 quando houver evidência
clara no texto do feedback.

7. FEED UP — CLAREZA SOBRE OS OBJETIVOS

Avalia se o feedback esclarece ou relembra o objetivo, o resultado esperado,
os critérios de sucesso ou o que deveria ser alcançado na atividade.

Score 0:
Não esclarece os objetivos ou critérios de sucesso.

Score 1:
Esclarece ou relembra os objetivos, critérios ou resultados esperados.

8. FEED BACK — PROGRESSO EM RELAÇÃO AO OBJETIVO

Avalia se o feedback informa como o desempenho atual do estudante se compara
ao objetivo ou ao resultado esperado.

Score 0:
Não informa o que foi alcançado ou o que ainda está distante do objetivo.

Score 1:
Explica o desempenho atual do estudante em relação aos objetivos ou critérios.

9. FEED FORWARD — PRÓXIMOS PASSOS

Avalia se o feedback indica ações futuras, próximos passos ou estratégias
para melhorar o trabalho ou a aprendizagem.

Score 0:
Não fornece orientação sobre o que fazer em seguida.

Score 1:
Apresenta ações, estratégias ou próximos passos para melhoria.

10. FEEDBACK SOBRE A TAREFA

Avalia se o feedback comenta diretamente a correção, completude, precisão
ou qualidade do produto final apresentado pelo estudante.

Pode incluir:
- identificação de respostas corretas ou incorretas;
- falta de informações;
- erros conceituais;
- adequação aos requisitos;
- qualidade do resultado final.

Score 0:
Não apresenta comentários sobre a execução ou o resultado da tarefa.

Score 1:
Apresenta comentários claros sobre a correção, qualidade ou completude da tarefa.

11. FEEDBACK SOBRE O PROCESSO

Avalia se o feedback comenta as estratégias, procedimentos, raciocínio
ou métodos utilizados para realizar a atividade.

Score 0:
Não aborda o processo utilizado pelo estudante.

Score 1:
Aborda como o estudante pensou, organizou, analisou ou executou a atividade.

12. FEEDBACK SOBRE AUTORREGULAÇÃO

Avalia se o feedback incentiva o estudante a monitorar, revisar, avaliar
ou controlar o próprio processo de aprendizagem.

Pode incluir:
- autoavaliação;
- revisão do próprio trabalho;
- planejamento;
- monitoramento da compreensão;
- verificação de erros;
- escolha de estratégias;
- reflexão sobre o próprio desempenho.

Score 0:
Não apresenta elementos relacionados à autorregulação.

Score 1:
Incentiva claramente a reflexão, o monitoramento ou a revisão autônoma.

13. FEEDBACK SOBRE O EU

Avalia se o feedback apresenta comentários pessoais dirigidos ao estudante
como indivíduo, em vez de se concentrar na tarefa, no processo ou na estratégia.

Exemplos:
- “Você é muito inteligente.”
- “Você é um excelente aluno.”
- “Você não leva jeito para isso.”

Score 0:
Não apresenta comentários pessoais sobre o estudante.

Score 1:
Apresenta elogios ou críticas dirigidos à pessoa, sem relação direta
com o desempenho específico na tarefa.

Atenção:
Não confunda um tom motivacional com feedback sobre o eu.
Comentários como “continue se esforçando” não são necessariamente feedback
sobre o eu, caso estejam vinculados à tarefa ou ao processo.

==================================================
PARTE 3 — ALUCINAÇÕES
==================================================

As dimensões de alucinação devem ser avaliadas de forma binária.

0 = a alucinação não foi identificada
1 = a alucinação foi identificada

Somente classifique como alucinação quando houver evidência clara.
Não considere simples diferenças de interpretação como alucinação.

14. ALUCINAÇÃO EM CONFLITO COM A ENTRADA

Avalia se o feedback contradiz, altera ou inventa informações sobre:

- a atividade;
- a rubrica;
- a resposta do estudante;
- dados presentes nos materiais fornecidos.

Score 0:
Não há informações que contradigam ou inventem elementos da entrada.

Score 1:
O feedback atribui ao estudante, à resposta ou à atividade alguma informação
que não existe ou que contradiz os dados fornecidos.

15. ALUCINAÇÃO EM CONFLITO COM O CONTEXTO

Avalia se o feedback contém afirmações internamente contraditórias.

Score 0:
O texto é internamente consistente.

Score 1:
O feedback apresenta duas ou mais afirmações incompatíveis entre si.

Exemplo:
Primeiro afirma que o estudante explicou corretamente um conceito e,
posteriormente, afirma que esse conceito não foi apresentado.

16. ALUCINAÇÃO EM CONFLITO COM FATOS

Avalia se o feedback contém afirmações que contradizem fatos objetivos,
conhecimentos consolidados ou informações verificáveis.

Score 0:
Não apresenta afirmações factualmente incorretas identificáveis.

Score 1:
Apresenta pelo menos uma afirmação claramente incompatível com fatos conhecidos.

Caso não seja possível verificar uma afirmação com segurança, atribua 0
e registre a incerteza na justificativa.

==================================================
REGRAS DE AVALIAÇÃO
==================================================

- Avalie somente o feedback fornecido.
- Use a descrição da atividade, a rubrica e a resposta do estudante como referência.
- Não premie o feedback apenas por ser longo.
- Não penalize o feedback apenas por ser curto.
- Não infira qualidades que não estejam explicitamente presentes.
- Não atribua uma dimensão com base apenas em uma frase genérica.
- As dimensões podem receber avaliações diferentes.
- Um feedback pode ter bom tom motivacional e baixa especificidade.
- Um feedback pode identificar erros corretamente, mas não apresentar feed forward.
- Um comentário pode ser simultaneamente feedback sobre tarefa e feed back.
- A presença de uma alucinação não implica automaticamente nota baixa
  em todas as outras dimensões.
- Não inclua texto fora do JSON solicitado.

==================================================
FORMATO DE SAÍDA
==================================================

Retorne exclusivamente um objeto JSON válido seguindo exatamente esta estrutura:

{
  "conteudo": {
    "alinhamento_objetivos": {
      "score": 0,
      "justificativa": ""
    },
    "especificidade": {
      "score": 0,
      "justificativa": ""
    },
    "tom_motivacional": {
      "score": 0,
      "justificativa": ""
    },
    "pontos_fortes": {
      "score": 0,
      "justificativa": ""
    },
    "pontos_fracos": {
      "score": 0,
      "justificativa": ""
    },
    "legibilidade": {
      "score": 0,
      "justificativa": ""
    }
  },
  "efetividade": {
    "feed_up": {
      "score": 0,
      "justificativa": ""
    },
    "feed_back": {
      "score": 0,
      "justificativa": ""
    },
    "feed_forward": {
      "score": 0,
      "justificativa": ""
    },
    "feedback_tarefa": {
      "score": 0,
      "justificativa": ""
    },
    "feedback_processo": {
      "score": 0,
      "justificativa": ""
    },
    "feedback_autorregulacao": {
      "score": 0,
      "justificativa": ""
    },
    "feedback_eu": {
      "score": 0,
      "justificativa": ""
    }
  },
  "alucinacoes": {
    "conflito_entrada": {
      "score": 0,
      "justificativa": ""
    },
    "conflito_contexto": {
      "score": 0,
      "justificativa": ""
    },
    "conflito_fatos": {
      "score": 0,
      "justificativa": ""
    }
  },
  "resumo_avaliacao": {
    "pontuacao_conteudo": 0,
    "pontuacao_maxima_conteudo": 12,
    "componentes_efetividade_presentes": 0,
    "total_componentes_efetividade": 7,
    "alucinacoes_detectadas": 0,
    "principais_observacoes": ""
  }
}
"""

In [7]:
prompts = []
for _, row in provas_sample.iterrows():
    prompts.append(
        rubric_eval_prompt
        .replace("{descricao_atividade}", str(row["question"]))
        .replace("{rubrica_atividade}", str(row["reference_answer"]))
        .replace("{resposta_estudante}", str(row["answer_text"]))
        .replace("{feedback}", str(row["feedback"]))
    )

print(f"Total de prompts gerados: {len(prompts)}")

Total de prompts gerados: 10


In [8]:
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY não encontrada. Confira .env e o diretório atual (os.getcwd()).")

genai.configure(api_key=api_key)
print("API Key configurada com sucesso:", api_key[:6] + "...")

API Key configurada com sucesso: AIzaSy...


In [9]:
print("CWD:", os.getcwd())
print("load_dotenv retornou:", load_dotenv())
print("GOOGLE_API_KEY existe?", os.getenv("GOOGLE_API_KEY") is not None)

CWD: c:\Users\anara\Desktop\Pesquisa - Tutoria\feedback-open-source\notebooks
load_dotenv retornou: True
GOOGLE_API_KEY existe? True


In [10]:
model = genai.GenerativeModel('models/gemini-2.5-flash')
generation_config = genai.GenerationConfig(
    response_mime_type="application/json"
)

In [11]:
cabecalhos = ["modelo","prompt_variant","question", "answer_text", "feedback", "analise_feedback"]

In [12]:
import json
import time
import random

In [13]:
def extrair_retry_delay(msg: str) -> float:
    """
    Extrai o tempo sugerido pelo servidor em mensagens do tipo 'retry in 12.34s'.
    Se não achar, retorna um fallback seguro.
    """
    match = re.search(r"retry in ([\d\.]+)s", msg)
    return float(match.group(1)) if match else 5.0

nome_arquivo_csv = 'prova-biologia-results-10-base-16dim.csv'
arquivo_existe = os.path.isfile(nome_arquivo_csv)

with open(nome_arquivo_csv, "a", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=cabecalhos)
    if not arquivo_existe:
        escritor.writeheader()

    for i, prompt_texto in enumerate(prompts):
        print(f"Processando prompt {i+1}/{len(prompts)}")

        tentativas_429 = 0

        while True:
            try:
                response = model.generate_content(
                    prompt_texto,
                    generation_config=generation_config
                )

                dados_avaliacao = json.loads(response.text)
                break  # saiu com sucesso

            except json.JSONDecodeError:
                print("JSON inválido. Reprocessando após 3s...")
                time.sleep(3)

            except Exception as e:
                msg = str(e)

                # tratamento de rate limit
                if "429" in msg:
                    tentativas_429 += 1

                    base_espera = extrair_retry_delay(msg)

                    # Buffer + jitter para não bater no limite exato da janela
                    buffer = 15
                    jitter = random.uniform(0, 5)

                    # Backoff leve se vierem vários 429 seguidos
                    backoff_extra = min(60, (tentativas_429 - 1) * 10)

                    espera = base_espera + buffer + jitter + backoff_extra

                    print(f"Rate limit atingido. Aguardando {espera:.1f}s...")
                    time.sleep(espera)

                    # Evita loop infinito em plano free
                    if tentativas_429 >= 8:
                        raise RuntimeError(
                            "Muitos 429 seguidos. Abortando para evitar loop infinito."
                        )

                else:
                    raise

        # pega a linha correspondente no dataframe
        linha = provas_sample.iloc[i]

        linha_dados = {
            "modelo": linha["modelo"],
            "prompt_variant": linha["prompt_variant"],
            "question": linha["question"],
            "answer_text": linha["answer_text"],
            "feedback": linha["feedback"],
            "analise_feedback": json.dumps(dados_avaliacao, ensure_ascii=False),
        }

        escritor.writerow(linha_dados)
        f.flush()

        print(f"✅ Dados salvos ({i+1}/{len(prompts)})")

        # cooldown conservador para o Gemini Free (60s às vezes ainda é pouco)
        time.sleep(30)


Processando prompt 1/10
✅ Dados salvos (1/10)
Processando prompt 2/10
✅ Dados salvos (2/10)
Processando prompt 3/10
✅ Dados salvos (3/10)
Processando prompt 4/10
✅ Dados salvos (4/10)
Processando prompt 5/10
✅ Dados salvos (5/10)
Processando prompt 6/10
✅ Dados salvos (6/10)
Processando prompt 7/10
✅ Dados salvos (7/10)
Processando prompt 8/10
✅ Dados salvos (8/10)
Processando prompt 9/10
✅ Dados salvos (9/10)
Processando prompt 10/10
✅ Dados salvos (10/10)
